In [ ]:
!pip install torch torchvision torch-geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.8 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn.functional as F
from torchvision.datasets import MNIST
from torchvision import transforms

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

import numpy as np
from tqdm.auto import trange


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cpu


In [ ]:
transform = transforms.ToTensor()

train_dataset = MNIST(root='.', train=True, download=True, transform=transform)
test_dataset  = MNIST(root='.', train=False, download=True, transform=transform)


100%|██████████| 9.91M/9.91M [00:00<00:00, 37.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 943kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.99MB/s]


In [ ]:

def image_to_graph(img, label):

    img = img.view(28,28)

    xs = []
    for i in range(28):
        for j in range(28):
            intensity = img[i,j]
            xs.append([intensity, i/27.0, j/27.0])

    x = torch.tensor(xs, dtype=torch.float)

    edges = []

    for i in range(28):
        for j in range(28):

            node = i*28 + j

            if i < 27:
                edges.append([node, (i+1)*28 + j])
                edges.append([(i+1)*28 + j, node])

            if j < 27:
                edges.append([node, i*28 + (j+1)])
                edges.append([i*28 + (j+1), node])

    edge_index = torch.tensor(edges, dtype=torch.long).t()

    return Data(x=x, edge_index=edge_index, y=torch.tensor(label))


In [ ]:
train_graphs = [image_to_graph(train_dataset[i][0], train_dataset[i][1])
                for i in range(5000)]
test_graphs = [image_to_graph(test_dataset[i][0], test_dataset[i][1])
               for i in range(1000)]

In [ ]:
train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_graphs, batch_size=32)

In [ ]:
class GCNNet(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = GCNConv(3,64)
        self.conv2 = GCNConv(64,128)
        self.conv3 = GCNConv(128,128)

        self.bn1 = torch.nn.BatchNorm1d(64)
        self.bn2 = torch.nn.BatchNorm1d(128)
        self.bn3 = torch.nn.BatchNorm1d(128)

        self.fc = torch.nn.Linear(128,10)

    def forward(self,data):

        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x,edge_index)
        x = self.bn1(x)
        x = F.relu(x)

        x = self.conv2(x,edge_index)
        x = self.bn2(x)
        x = F.relu(x)

        x = self.conv3(x,edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        x = global_mean_pool(x,batch)

        x = F.dropout(x,p=0.5,training=self.training)

        return self.fc(x)


In [ ]:
model = GCNNet().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)


In [ ]:
def train():

    model.train()
    total_loss = 0

    for data in train_loader:

        data = data.to(device)

        optimizer.zero_grad()

        out = model(data)

        loss = F.cross_entropy(out,data.y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


In [ ]:
@torch.no_grad()
def test(loader):

    model.eval()

    correct = 0

    for data in loader:

        data = data.to(device)

        out = model(data)

        pred = out.argmax(dim=1)

        correct += int((pred == data.y).sum())

    return correct / len(loader.dataset)


In [ ]:
epochs = 30

for epoch in trange(1,epochs+1):

    loss = train()

    acc = test(test_loader)

    print(f"Epoch {epoch:02d} | Loss {loss:.4f} | Test Acc {acc:.4f}")


  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 01 | Loss 2.1260 | Test Acc 0.3800
Epoch 02 | Loss 1.7795 | Test Acc 0.5180
Epoch 03 | Loss 1.5191 | Test Acc 0.4690
Epoch 04 | Loss 1.3399 | Test Acc 0.6080
Epoch 05 | Loss 1.2344 | Test Acc 0.6920
Epoch 06 | Loss 1.1544 | Test Acc 0.7170
Epoch 07 | Loss 1.0993 | Test Acc 0.6850
Epoch 08 | Loss 1.0180 | Test Acc 0.7750
Epoch 09 | Loss 0.9956 | Test Acc 0.7230
Epoch 10 | Loss 0.9490 | Test Acc 0.6040
Epoch 11 | Loss 0.9280 | Test Acc 0.7400
Epoch 12 | Loss 0.8885 | Test Acc 0.7980
Epoch 13 | Loss 0.8832 | Test Acc 0.7530
Epoch 14 | Loss 0.8498 | Test Acc 0.8020
Epoch 15 | Loss 0.8351 | Test Acc 0.7150
Epoch 16 | Loss 0.8212 | Test Acc 0.8050
Epoch 17 | Loss 0.8031 | Test Acc 0.8080
Epoch 18 | Loss 0.7750 | Test Acc 0.8170
Epoch 19 | Loss 0.7650 | Test Acc 0.8140
Epoch 20 | Loss 0.7523 | Test Acc 0.8260
Epoch 21 | Loss 0.7462 | Test Acc 0.7920
Epoch 22 | Loss 0.7253 | Test Acc 0.8210
Epoch 23 | Loss 0.7177 | Test Acc 0.8120
Epoch 24 | Loss 0.6893 | Test Acc 0.8220
Epoch 25 | Loss 